# 02 - Next Day Prediction

Predict tomorrow's ETHUSDT close.


## Setup

In [1]:
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from analysis.loader import load_ohlcv
from analysis.indicators import add_all

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 4)
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
FEATURES = ["ret_1","ret_5","ret_20","vol_20","rsi_14","macd_hist","atr_14","sma_50","sma_200"]

def build(timeframe="1d", horizon=1):
    df = load_ohlcv(timeframe=timeframe)
    df = add_all(df).copy()
    df["ret_1"]  = df["close"].pct_change(1)
    df["ret_5"]  = df["close"].pct_change(5)
    df["ret_20"] = df["close"].pct_change(20)
    df["vol_20"] = df["ret_1"].rolling(20).std()
    df["target"] = df["close"].pct_change(horizon).shift(-horizon)
    return df.dropna()

def metrics(y, p):
    return {"rmse": float(np.sqrt(((y-p)**2).mean())),
            "mae":  float(np.abs(y-p).mean()),
            "dir_accuracy": float(((p>0)==(y>0)).mean())}

## 1. Load daily

In [2]:
df = build()
print("rows:", len(df))
df[["close","ret_1","vol_20","target"]].tail()

rows: 1989


,close,ret_1,vol_20,target
date,,,,
2026-09-27,2688.65,-0.002848,0.024805,0.000022
2026-09-28,2688.71,0.000022,0.024782,-0.003991
2026-09-29,2677.98,-0.003991,0.024724,0.002999
2026-09-30,2686.01,0.002999,0.024423,0.007599
2026-10-01,2706.42,0.007599,0.023603,-0.013978


## 2. Fit Ridge

In [3]:
train = df.iloc[:-250]; test = df.iloc[-250:]
model = Ridge(alpha=1.0).fit(train[FEATURES].values, train["target"].values)
pred = model.predict(test[FEATURES].values); y = test["target"].values
print(json.dumps(metrics(y, pred), indent=2))

{
  "rmse": 0.03278088130017453,
  "mae": 0.02187720379544318,
  "dir_accuracy": 0.532
}


## 3. Forecast tomorrow

In [4]:
r = float(model.predict(df[FEATURES].iloc[[-1]].values)[0])
last = float(df["close"].iloc[-1]); rs = float((y-pred).std())
lo, hi = last*(1+r-1.28*rs), last*(1+r+1.28*rs)
print(f"as of: {df.index[-1].date()}  last: {last:.2f}")
print(f"pred ret: {r*100:+.4f}%   pred close: {last*(1+r):.2f}")
print(f"80% interval: [{lo:.2f}, {hi:.2f}]")
(OUT/"next_day.json").write_text(json.dumps(
    {"as_of": str(df.index[-1].date()), "last_close": last,
     "next_ret": r, "next_close": last*(1+r), "p10": lo, "p90": hi}, indent=2))
print("wrote next_day.json")

as of: 2026-10-01  last: 2706.42
pred ret: +0.6254%   pred close: 2723.34
80% interval: [2609.93, 2836.76]
wrote next_day.json
